# ScArlet-Sails — Kaggle Smoke Test

Запускает на Kaggle:
1. Клон репозитория (ветка с ревизией 2026-05).
2. Установка зависимостей (`requirements.txt` — vectorbt, quantstats, xgboost, FAISS).
3. Прогон pytest-сьюта (синтетические OHLCV-фикстуры подтянутся автоматически через `tests/conftest.py`).
4. Smoke-test нового vectorbt-движка: `RSI` стратегия на синтетическом BTC/15m, плюс `combined` на 4-х монетах.

**Перед запуском:** убедись, что ветка с ревизией запушена в GitHub. По умолчанию используется `claude/quizzical-raman-434cfb`; меняй переменную `BRANCH` если другая.

**Аппаратура:** Kaggle CPU достаточно. GPU не нужен (vectorbt векторизован, XGBoost CPU).

In [ ]:
REPO_URL = 'https://github.com/StarDust1508/ScArlet-Sails.git'
BRANCH = 'claude/quizzical-raman-434cfb'
PROJECT_DIR = '/kaggle/working/ScArlet-Sails'

## 1. Clone

In [ ]:
import os, subprocess, shutil, sys

if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.check_call([
    'git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, PROJECT_DIR
])
os.chdir(PROJECT_DIR)
print('cwd =', os.getcwd())
print('files =', sorted(os.listdir(PROJECT_DIR))[:20])

## 2. Install dependencies

vectorbt тянет numba; первая установка ~3-5 мин.

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -20

## 3. Run pytest

`tests/conftest.py` создаст синтетические parquet-фикстуры в `data/raw/`, чтобы
тесты `test_data_loader.py` работали без DVC-данных. Deprecated/research-тесты
(Q-learner, RAG e2e, 74-col spec без реальных файлов) помечены skip.

In [ ]:
!pytest tests/ --tb=short -q 2>&1 | tail -60

## 4. Smoke vectorbt engine

Запускаем `VBTBacktestEngine` на синтетических данных — проверяем, что
новый движок жив, метрики считаются.

In [ ]:
sys.path.insert(0, PROJECT_DIR)

from backtesting.vbt_engine import VBTBacktestEngine, VBTBacktestConfig
from strategies.simple_strategies import SimpleRSIStrategy, CombinedStrategy
from core.ood_detector import OODDetector

engine = VBTBacktestEngine(VBTBacktestConfig(
    initial_capital=10_000,
    commission=0.001,
    slippage=0.0005,
    stop_loss_pct=0.02,
))

result = engine.run(SimpleRSIStrategy(), coin='BTC', timeframe='15m')
print(result.summary())

In [ ]:
# Multi-asset comparison
from backtesting.vbt_engine import run_multi_asset

results = run_multi_asset(
    engine,
    strategy_factory=lambda: CombinedStrategy(ood_detector=OODDetector()),
    coins=['BTC', 'ETH', 'SOL', 'AVAX'],
    timeframe='15m',
)

import pandas as pd
df = pd.DataFrame({coin: r.metrics for coin, r in results.items()}).T
df[['total_return_pct', 'sharpe', 'max_drawdown_pct', 'num_trades', 'win_rate_pct']]

## 5. Что искать в выводе

- **pytest**: ожидание ~120-140 passed, остальное skip (Q-learner, RAG e2e). Падений быть не должно.
- **VBT-смоук**: на синтетике Sharpe будет случайный (это GBM), важно что движок отрабатывает без exception, `num_trades > 0`.
- **Multi-asset**: метрики приходят в DataFrame, никаких NaN-only строк.

Если что-то ломается — присылай stderr/traceback, докручиваем.